# 강남구 도로 위 PM10 시각화
V5 ST-GNN ambient PM10 예측값을 강남구 도로망 위에 표시합니다.

**셀 3의 `TARGET_DT` 만 바꾸고 전체 실행하세요.**

In [ ]:
%matplotlib inline
import os, sys, struct, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from matplotlib.collections import LineCollection
from scipy.spatial import cKDTree
warnings.filterwarnings('ignore')

plt.rcParams['font.family']       = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 경로 ──────────────────────────────────────────────────────────────────
ROOT     = '/workspace/ST-GNN Modeling'
OUT      = os.path.join(ROOT, 'Visualization/outputs')
GRID_CSV = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/격자_250m_4326.csv'
GPKG     = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/서울 도로 위계 및 GVI.gpkg'
V5_TRAIN = os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_train.npy')
V5_VAL   = os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_val.npy')
V5_TEST  = os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_test.npy')
TS_LOOKUP = os.path.join(ROOT, 'RoadExtension_V2/checkpoints/v5_ts_lookup.csv')
os.makedirs(OUT, exist_ok=True)

# 강남구 바운딩 박스 (시각화 범위)
BBOX = dict(lat_min=37.485, lat_max=37.540,
            lon_min=127.005, lon_max=127.100)

print('설정 완료')

In [ ]:
# ── 데이터 로드 (1회) ─────────────────────────────────────────────────────

# V5 격자 + PM10
grid_df   = pd.read_csv(GRID_CSV)
v5_data   = {'train': np.load(V5_TRAIN),
             'val':   np.load(V5_VAL),
             'test':  np.load(V5_TEST)}
ts_lookup = pd.read_csv(TS_LOOKUP)
ts_lookup['dt'] = pd.to_datetime(ts_lookup['timestamp'])

# 격자 KDTree (PM10 보간용)
grid_coords = grid_df[['lat', 'lon']].values.astype(float)
grid_tree   = cKDTree(grid_coords)

# ── WKB LineString 파서 ────────────────────────────────────────────────────
def parse_gpkg_linestring(data):
    """GeoPackage WKB → [(lon, lat), ...] 좌표 목록"""
    if not data or data[0:2] != b'GP': return []
    flags  = data[3]
    env_sz = {0:0, 1:32, 2:48, 3:48, 4:64}.get((flags>>1)&7, 0)
    wkb    = data[8 + env_sz:]
    if len(wkb) < 9: return []
    endian = '<' if wkb[0] == 1 else '>'
    gtype  = struct.unpack_from(endian+'I', wkb, 1)[0] & 0x1FFFFFFF
    if gtype not in (2, 5): return []
    off = 5
    if gtype == 5:
        n = struct.unpack_from(endian+'I', wkb, off)[0]
        if n == 0: return []
        off += 4
        endian = '<' if wkb[off]==1 else '>'
        off += 5
    n_pts = struct.unpack_from(endian+'I', wkb, off)[0]; off += 4
    pts = []
    for _ in range(n_pts):
        x, y = struct.unpack_from(endian+'dd', wkb, off)
        pts.append((x, y)); off += 16
    return pts

# ── 강남구 도로 로드 ──────────────────────────────────────────────────────
print('강남구 도로 로드 중...')
conn = sqlite3.connect(GPKG)
cur  = conn.cursor()
cur.execute('SELECT "geom", "highway_type", "lanes_num" FROM "서울 도로 위계 및 GVI"')
rows = cur.fetchall()
conn.close()

roads = []
for geom, htype, lanes in rows:
    pts = parse_gpkg_linestring(geom)
    if not pts: continue
    lons = [p[0] for p in pts]
    lats = [p[1] for p in pts]
    mid_lon, mid_lat = np.mean(lons), np.mean(lats)
    if (BBOX['lat_min'] <= mid_lat <= BBOX['lat_max'] and
        BBOX['lon_min'] <= mid_lon <= BBOX['lon_max']):
        roads.append({'pts': pts, 'htype': htype,
                      'lanes': lanes or 2.0})

# 도로 타입별 선 굵기
LW_MAP = {'trunk': 3.5, 'primary': 2.8,
          'secondary': 2.0, 'trunk_link': 1.4,
          'secondary_link': 1.0}

print('강남구 도로 구간: {:,}개'.format(len(roads)))
print('V5 test 기간: {} ~ {}'.format(
    ts_lookup[ts_lookup['split']=='test']['dt'].min().strftime('%Y-%m-%d'),
    ts_lookup[ts_lookup['split']=='test']['dt'].max().strftime('%Y-%m-%d')))

In [ ]:
# ── 시각화 ───────────────────────────────────────────────────────────────
from matplotlib.ticker import FuncFormatter

# 하늘색 계열 컬러맵 (낮음=흰/연청, 높음=진파랑)
CMAP = mcolors.LinearSegmentedColormap.from_list(
    'skyblue_pm',\n    ['#FFFFFF', '#C9E8FF', '#6BAED6', '#2171B5', '#08306B'])\n\nnorm  = mcolors.Normalize(vmin=vmin, vmax=vmax)\n\nBG = '#F7F7F5'\n\nfig, ax = plt.subplots(figsize=(9, 9))\nfig.patch.set_facecolor(BG)\nax.set_facecolor(BG)\n\nax.add_patch(FancyBboxPatch(\n    (BBOX['lon_min'], BBOX['lat_min']),\n    BBOX['lon_max'] - BBOX['lon_min'],\n    BBOX['lat_max'] - BBOX['lat_min'],\n    boxstyle='square,pad=0',\n    facecolor='#FAFAFA', edgecolor='none', zorder=0))\n\n# 도로 타입별로 그리기 (하위 → 상위 순)\ntype_order = ['secondary_link', 'trunk_link', 'secondary', 'primary', 'trunk']\nfor htype in type_order:\n    lw_base = LW_MAP.get(htype, 1.2)\n    subset  = [r for r in roads if r['htype'] == htype]\n    if not subset: continue\n    for r in subset:   # 그림자\n        xs = [p[0] for p in r['pts']]; ys = [p[1] for p in r['pts']]\n        ax.plot(xs, ys, color='#E0E0E0', lw=lw_base+1.4,\n                solid_capstyle='round', solid_joinstyle='round', zorder=1, alpha=0.55)\n    for r in subset:   # 색상 선\n        xs = [p[0] for p in r['pts']]; ys = [p[1] for p in r['pts']]\n        ax.plot(xs, ys, color=CMAP(norm(r['pm10'])), lw=lw_base,\n                solid_capstyle='round', solid_joinstyle='round', zorder=2, alpha=0.95)\n\n# 축 — 실제 위경도 값 표시\nax.set_xlim(BBOX['lon_min'], BBOX['lon_max'])\nax.set_ylim(BBOX['lat_min'], BBOX['lat_max'])\nax.set_aspect('equal', adjustable='box')\nax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: '{:.3f}'.format(x)))\nax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:.3f}'.format(y)))\nax.set_xlabel('경도 (Longitude)', fontsize=10, color='#555555')\nax.set_ylabel('위도 (Latitude)',  fontsize=10, color='#555555')\nax.tick_params(colors='#888888', labelsize=8)\nfor sp in ax.spines.values():\n    sp.set_edgecolor('#CCCCCC'); sp.set_linewidth(0.7)\nax.grid(True, color='#EEEEEE', linewidth=0.5, alpha=0.6)\n\n# 컬러바\nsm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm)\nsm.set_array([])\ncbar = fig.colorbar(sm, ax=ax, fraction=0.033, pad=0.02, aspect=28)\ncbar.set_label('PM10 (μg/m³)', fontsize=10, color='#333333')\ncbar.ax.tick_params(labelsize=8.5, colors='#555555')\ncbar.outline.set_edgecolor('#CCCCCC')\n\n# 제목\nax.set_title(\n    '강남구 도로별 Ambient PM10  |  {}'.format(actual_dt.strftime('%Y-%m-%d  %H:00')),\n    fontsize=13, fontweight='bold', color='#1A1A1A', pad=12)\n\n# 통계\nax.text(0.02, 0.02,\n        'mean={:.1f}  max={:.1f} μg/m³   |   {:,}개 구간'.format(\n            pm_values.mean(), pm_values.max(), len(roads)),\n        transform=ax.transAxes, fontsize=8, color='#555555',\n        va='bottom', ha='left',\n        bbox=dict(facecolor='white', edgecolor='#CCCCCC',\n                  alpha=0.88, boxstyle='round,pad=0.4'))\n\nplt.tight_layout()\n\nout = os.path.join(OUT, 'gangnam_road_pm10_{}.png'.format(\n    actual_dt.strftime('%Y%m%d_%H00')))\nplt.savefig(out, dpi=200, bbox_inches='tight',\n            facecolor=BG, pad_inches=0.12)\nprint('저장:', out)\nplt.show()

In [ ]:
# ── PM10 획득 + 도로 매핑 ─────────────────────────────────────────────────
dt    = pd.Timestamp(TARGET_DT)
match = ts_lookup[ts_lookup['dt'] == dt]
if len(match) == 0:
    match = ts_lookup.loc[[(ts_lookup['dt'] - dt).abs().idxmin()]]
row       = match.iloc[0]
pm10_all  = v5_data[row['split']][int(row['local_idx'])]
actual_dt = pd.Timestamp(row['dt'])

# 각 도로 구간의 각 포인트마다 가장 가까운 격자 PM10 조회
def road_pm10(pts):
    """도로 구간 좌표들의 평균 PM10 반환."""
    latlon = [(p[1], p[0]) for p in pts]   # (lat, lon)
    _, idxs = grid_tree.query(latlon)
    return float(pm10_all[idxs].mean())

print('도로별 PM10 계산 중...')
for r in roads:
    r['pm10'] = road_pm10(r['pts'])

pm_values = np.array([r['pm10'] for r in roads])
vmin, vmax = pm_values.min(), pm_values.max()
print('PM10 범위: {:.2f} ~ {:.2f} μg/m³  (mean={:.2f})'.format(
    vmin, vmax, pm_values.mean()))

In [ ]:
# ── 시각화 ───────────────────────────────────────────────────────────────
CMAP = plt.cm.get_cmap('YlOrRd')   # 낮음=노랑, 높음=빨강 (포스터용)
# 하늘색 계열 원할 시: 아래 줄 주석 해제
# CMAP = plt.cm.get_cmap('Blues')

norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)

BG    = '#F5F5F0'     # 연회색 배경 (지도 느낌)
WHITE = '#FFFFFF'

fig, ax = plt.subplots(figsize=(9, 9))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# 배경 구역 (연한 흰색 박스)
from matplotlib.patches import FancyBboxPatch
ax.add_patch(FancyBboxPatch(
    (BBOX['lon_min'], BBOX['lat_min']),
    BBOX['lon_max'] - BBOX['lon_min'],
    BBOX['lat_max'] - BBOX['lat_min'],
    boxstyle='square,pad=0',
    facecolor='#FAFAF8', edgecolor='none', zorder=0))

# 도로 타입별로 그리기 (trunk → primary → secondary 순)
type_order = ['secondary_link', 'trunk_link', 'secondary', 'primary', 'trunk']
for htype in type_order:
    lw_base = LW_MAP.get(htype, 1.2)
    subset  = [r for r in roads if r['htype'] == htype]
    if not subset: continue

    # 도로 그림자 (약간 두꺼운 회색 선 먼저)
    for r in subset:
        xs = [p[0] for p in r['pts']]
        ys = [p[1] for p in r['pts']]
        ax.plot(xs, ys, color='#DDDDDD',
                lw=lw_base + 1.2, solid_capstyle='round',
                solid_joinstyle='round', zorder=1, alpha=0.6)

    # 실제 PM10 색상 선
    for r in subset:
        xs = [p[0] for p in r['pts']]
        ys = [p[1] for p in r['pts']]
        color = CMAP(norm(r['pm10']))
        ax.plot(xs, ys, color=color,
                lw=lw_base, solid_capstyle='round',
                solid_joinstyle='round', zorder=2, alpha=0.92)

# 축 설정
ax.set_xlim(BBOX['lon_min'], BBOX['lon_max'])
ax.set_ylim(BBOX['lat_min'], BBOX['lat_max'])
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Longitude', fontsize=10, color='#555555')
ax.set_ylabel('Latitude',  fontsize=10, color='#555555')
ax.tick_params(colors='#888888', labelsize=8.5)
for sp in ax.spines.values():
    sp.set_edgecolor('#CCCCCC'); sp.set_linewidth(0.8)
ax.grid(True, color='#EEEEEE', linewidth=0.5, alpha=0.7)

# 컬러바
sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.033, pad=0.02, aspect=28)
cbar.set_label('PM10 (μg/m³)', fontsize=10, color='#333333')
cbar.ax.tick_params(labelsize=8.5, colors='#555555')
cbar.outline.set_edgecolor('#CCCCCC')

# 제목
ax.set_title(
    '강남구 도로별 Ambient PM10\n{}'.format(actual_dt.strftime('%Y-%m-%d  %H:00')),
    fontsize=13, fontweight='bold', color='#222222', pad=12)

# 통계 텍스트
ax.text(0.02, 0.02,
        'mean={:.1f}  max={:.1f} μg/m³\n도로 구간 {:,}개'.format(
            pm_values.mean(), pm_values.max(), len(roads)),
        transform=ax.transAxes, fontsize=8, color='#666666',
        va='bottom', ha='left',
        bbox=dict(facecolor='white', edgecolor='#CCCCCC',
                  alpha=0.85, boxstyle='round,pad=0.4'))

plt.tight_layout()

out = os.path.join(OUT, 'gangnam_road_pm10_{}.png'.format(
    actual_dt.strftime('%Y%m%d_%H00')))
plt.savefig(out, dpi=200, bbox_inches='tight',
            facecolor=BG, pad_inches=0.12)
print('저장:', out)
plt.show()